# Session 3: cooling a eutectic alloy, and when equilibrium fails
**A. Real-world problem.** Electronics solder melts at a low temperature because tin and lead
(the classic recipe) form a *eutectic*: a mixture that melts lower than either metal. When a
solder joint cools in a factory it may freeze in a fraction of a second, far too fast for the atoms to
reach the equilibrium that the lever rule assumes. The joint then contains a different structure than the diagram
predicts. Here we (1) build a eutectic diagram, (2) simulate the cooling curve with latent heat,
(3) compare equilibrium with fast (Scheil) freezing, and (4) find where the equilibrium assumption fails.

**Illustrative numbers.** The diagram is *Pb-Sn-like*, with rounded round-number values (lead side
327 C, tin side 232 C, eutectic 183 C at 62 wt% Sn, solubility limits 19 and 97 wt% Sn) and straight lines.
The heat capacity, latent heat and diffusion data are illustrative, not measured design data.
Composition x is weight percent Sn. Temperatures in C, time in s, heat in J/g.

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

T_PB, T_SN, T_E = 327.0, 232.0, 183.0       # melting points of the pure metals and the eutectic, C
C_ALPHA, C_E, C_BETA = 19.0, 62.0, 97.0     # alpha solubility limit, eutectic point, beta limit (wt% Sn)
M_L = (T_PB - T_E) / C_E                     # liquidus slope on the lead side, C per wt% Sn
M_S = (T_PB - T_E) / C_ALPHA                 # solidus slope on the lead side, C per wt% Sn
K = C_ALPHA / C_E                            # partition coefficient C_solid / C_liquid (constant here)

C_HEAT = 0.2          # specific heat, J/(g K), same in every phase (simplification)
L_HEAT = 50.0         # latent heat of freezing, J/g (illustrative)
D_SOLID = 1.0e-12     # diffusion coefficient of Sn in solid alpha, m^2/s (illustrative)
LAMBDA = 20.0e-6      # half the dendrite arm spacing, m (illustrative; really depends on cooling rate)

## Step 1: the eutectic diagram (lead-rich side)
Liquidus: T_L(x) = 327 - M_L x.  Solidus of the alpha solid: T_S(x) = 327 - M_S x for x <= 19.
Both lines start at the same point (pure Pb), so the ratio of the compositions on a tie line is
*constant*: C_solid / C_liquid = M_L / M_S = 19/62 = K (about 0.31). Below 183 C the liquid left
(62 wt% Sn) freezes all at once into a fine eutectic mixture of alpha and beta.

In [ ]:
def t_liquidus(x):
    return T_PB - M_L * x


def t_solidus(x):
    """Equilibrium end of freezing: the alpha solidus, or the eutectic line when x > 19."""
    return max(T_PB - M_S * x, T_E) if x < C_E else T_E


def eq_solid_fraction(T, x):
    """Equilibrium (lever rule) solid fraction at T for a lead-rich alloy x <= 62 wt% Sn.
    Above the liquidus: 0. Just above 183 C for x > 19 it is (62 - x)/(62 - 19); at or below
    the end of freezing it is 1 (the eutectic freezes at 183 C)."""
    tl, ts = t_liquidus(x), t_solidus(x)
    if tl - ts < 1e-9:                     # pure Pb or the eutectic composition
        return 0.0 if T > tl + 1e-9 else 1.0
    if T >= tl:
        return 0.0
    if T <= ts:
        return 1.0
    c_l = (T_PB - T) / M_L
    c_s = (T_PB - T) / M_S
    return (c_l - x) / (c_l - c_s)


def pro_eutectic_fraction(x):
    """Fraction of primary alpha (grown before the eutectic) just above 183 C, for 19 <= x <= 62."""
    return (C_E - x) / (C_E - C_ALPHA)


def eutectic_mixture_fraction(x):
    return 1.0 - pro_eutectic_fraction(x)


def liquid_fraction_eq(x, T):
    return 1.0 - eq_solid_fraction(T, x)

## Step 2: a cooling simulation with latent heat (the enthalpy method)
Heat content per gram: H(T) = c T + L (1 - f_s(T)).  While heat leaves at rate q (W/g),

    dH/dt = -q      so      H_(n+1) = H_n - q dt,   then find T with H(T) = H_(n+1).

f_s(T) is the lever-rule solid fraction. Where f_s changes quickly with T the temperature falls
slowly (the *effective* heat capacity is c + L |d f_s / dT|); where f_s jumps (the eutectic) the
temperature stops: a flat arrest. This is energy conservation applied step by step.

In [ ]:
def temperature_from_enthalpy(H, fs_fun, lo=0.0, hi=700.0):
    """Invert H(T) = c T + L (1 - fs_fun(T)) by bisection. Works across jumps (arrests)."""
    a, b = lo, hi
    for _ in range(55):
        m = 0.5 * (a + b)
        if C_HEAT * m + L_HEAT * (1.0 - fs_fun(m)) >= H:
            b = m
        else:
            a = m
    return 0.5 * (a + b)


def simulate_cooling(fs_fun, q_fun, T0, T_end, n_steps=3000, max_steps=200000):
    """March the enthalpy forward in time. q_fun(T) is the heat loss rate (W/g) at temperature T.
    Returns arrays t (s), T (C) and the frozen fraction f (from f = 1 - (H - cT)/L)."""
    H = C_HEAT * T0 + L_HEAT * (1.0 - fs_fun(T0))
    H_end = C_HEAT * T_end + L_HEAT
    dt = (H - H_end) / q_fun(T0) / n_steps
    ts, Ts, fs = [0.0], [T0], [fs_fun(T0)]
    T = T0
    while T > T_end and len(ts) < max_steps:
        H -= q_fun(T) * dt
        T = temperature_from_enthalpy(H, fs_fun)
        ts.append(ts[-1] + dt)
        Ts.append(T)
        fs.append(min(1.0, max(0.0, 1.0 - (H - C_HEAT * T) / L_HEAT)))
    return np.array(ts), np.array(Ts), np.array(fs)


def constant_q(q):
    return lambda T: q


def newtonian_q(h, T_amb=25.0):
    """Heat loss q = h (T - T_amb) with h in W/(g K)."""
    return lambda T: h * (T - T_amb)


def arrest_duration(t, T, T_level, tol=1e-6):
    """Time the temperature sits within tol of T_level (flat arrest)."""
    dt = t[1] - t[0]
    return float(np.sum(np.abs(T - T_level) < tol) * dt)


def time_to_reach(t, T, T_target):
    """First time the falling curve reaches T_target (linear interpolation)."""
    i = int(np.argmax(T <= T_target))
    if i == 0:
        return float(t[0])
    f = (T[i - 1] - T_target) / (T[i - 1] - T[i])
    return float(t[i - 1] + f * (t[i] - t[i - 1]))

## Step 3: Scheil (no diffusion in the solid) versus equilibrium
At the other extreme from the lever rule, assume the solid does not change after it forms
and the liquid stays perfectly mixed. A thin slice dfs of solid has composition k C_L. Sn leaving
the liquid: (1 - f_s) dC_L = (C_L - k C_L) d f_s, which integrates to the **Scheil equation**

    C_L = C0 f_L^(k-1),     C_S = k C_L = k C0 f_L^(k-1),     f_L = 1 - f_s.

The liquid keeps getting richer in Sn until it reaches the eutectic 62 wt% Sn, even for alloys
the equilibrium diagram says are fully solid before that.

In [ ]:
def scheil_liquid_composition(fl, x, k=K):
    return x * np.asarray(fl, dtype=float) ** (k - 1.0)


def scheil_solid_composition(fl, x, k=K):
    return k * scheil_liquid_composition(fl, x, k)


def scheil_eutectic_fraction(x, k=K):
    """Liquid fraction left when C_L reaches the eutectic 62 wt% Sn (it then freezes as eutectic)."""
    return min(1.0, (C_E / x) ** (1.0 / (k - 1.0)))


def scheil_solid_fraction(T, x, k=K):
    """Scheil f_s at temperature T (C) for x < 62: the eutectic arrest takes all liquid left at 183 C."""
    if T >= t_liquidus(x):
        return 0.0
    if T <= T_E:
        return 1.0
    fl = ((T_PB - T) / (M_L * x)) ** (1.0 / (k - 1.0))
    return 1.0 - fl


def scheil_by_integration(x, fs_end, k=K, n=20000):
    """Integrate the mass-balance equation dC_L/d f_s = (1 - k) C_L / (1 - f_s) with RK4."""
    h = fs_end / n
    c, f = x, 0.0

    for _ in range(n):
        k1 = (1.0 - k) * c / (1.0 - f)
        k2 = (1.0 - k) * (c + 0.5 * h * k1) / (1.0 - f - 0.5 * h)
        k3 = (1.0 - k) * (c + 0.5 * h * k2) / (1.0 - f - 0.5 * h)
        k4 = (1.0 - k) * (c + h * k3) / (1.0 - f - h)
        c += h * (k1 + 2 * k2 + 2 * k3 + k4) / 6.0
        f += h
    return c


def scheil_mass_balance(x, fs_end, k=K, n=200001):
    """Total Sn found in the solid made so far plus the Sn left in the liquid, divided by x.
    Must equal 1 (conservation of Sn)."""
    fs = np.linspace(0.0, fs_end, n)
    cs = scheil_solid_composition(1.0 - fs, x, k)
    sn_solid = float(np.sum(0.5 * (cs[1:] + cs[:-1]) * np.diff(fs)))
    sn_liquid = (1.0 - fs_end) * float(scheil_liquid_composition(1.0 - fs_end, x, k))
    return (sn_solid + sn_liquid) / x

## Step 4: in between (Clyne-Kurz back-diffusion) and the effect of cooling rate
Real freezing lies between the two extremes. The Clyne-Kurz model adds limited diffusion in the solid
through one number, alpha = D t_f / lambda^2 (diffusion distance squared over the arm size squared,
with t_f the local freezing time). Slow cooling = long t_f = large alpha = lever rule.
Fast cooling = tiny alpha = Scheil. The model uses

    a' = alpha (1 - exp(-1/alpha)) - 0.5 exp(-1/(2 alpha)),     C_L = C0 [1 - (1 - 2 a' k) f_s]^((k-1)/(1 - 2 a' k)).

a' = 0 gives Scheil and a' = 0.5 gives the lever rule exactly.

In [ ]:
def alpha_prime(alpha):
    if alpha <= 0.0:
        return 0.0
    if alpha > 1e3:
        return 0.5                      # lever-rule limit (avoids round-off for huge alpha)
    return alpha * (1.0 - math.exp(-1.0 / alpha)) - 0.5 * math.exp(-1.0 / (2.0 * alpha))


def clyne_kurz_liquid_composition(fs, x, ap, k=K):
    s = 1.0 - 2.0 * ap * k
    return x * (1.0 - s * np.asarray(fs, dtype=float)) ** ((k - 1.0) / s)


def clyne_kurz_eutectic_fraction(x, ap, k=K):
    """Fraction of the casting that freezes as eutectic when back-diffusion strength is a'."""
    s = 1.0 - 2.0 * ap * k
    fs_at_e = (1.0 - (C_E / x) ** (s / (k - 1.0))) / s
    return max(0.0, 1.0 - fs_at_e)


def back_diffusion_alpha(t_freeze):
    return D_SOLID * t_freeze / LAMBDA ** 2


def freezing_time(x, q):
    """Time (s) to cross the freezing range at constant heat loss q (W/g), equilibrium freezing, from the simulation."""
    tl = t_liquidus(x)
    ts = t_solidus(x)
    t, T, _ = simulate_cooling(lambda T_: eq_solid_fraction(T_, x), constant_q(q), tl + 20.0, ts - 20.0, n_steps=4000)
    return time_to_reach(t, T, ts + 1e-9) - time_to_reach(t, T, tl - 1e-9)

## Step 5: a design question for the mini-project
Which alloy, and which temperature, gives a wanted liquid fraction (a pasty solder)?
For 19 < x < 62 the liquid fraction between 183 C and the liquidus follows the lever rule.

In [ ]:
def temperature_for_liquid_fraction(x, target):
    """Temperature (C) in the alpha + liquid band where alloy x (19 < x < 62) has liquid fraction `target`.
    If the target is below the liquid fraction available just above 183 C, that fraction is only
    reached during the eutectic arrest, so 183 C is returned."""
    lo, hi = T_E + 1e-9, t_liquidus(x)
    if liquid_fraction_eq(x, lo) >= target:
        return T_E
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        if liquid_fraction_eq(x, mid) > target:
            hi = mid
        else:
            lo = mid
    return 0.5 * (lo + hi)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    # --- figure 1: the diagram with two alloys
    xa = np.linspace(0, C_E, 100)
    xb = np.linspace(C_E, 100, 100)
    fig, ax = plt.subplots(figsize=(6.6, 4.8))
    ax.plot(xa, T_PB - M_L * xa, "r-", label="liquidus")
    ax.plot(xb, T_SN + (T_E - T_SN) * (100 - xb) / (100 - C_E), "r-")
    ax.plot(np.linspace(0, C_ALPHA, 30), T_PB - M_S * np.linspace(0, C_ALPHA, 30), "b-", label="solidus")
    ax.plot([C_BETA, 100], [T_E, T_SN], "b-")
    ax.plot([0, 100], [T_E, T_E], "b-")
    for xx, lab in ((10, "10"), (40, "40")):
        ax.axvline(xx, color="gray", ls=":")
        ax.text(xx + 1, 345, lab + " wt% Sn", fontsize=8)
    ax.set_xlabel("composition (weight percent Sn)")
    ax.set_ylabel("temperature (C)")
    ax.set_title("Illustrative Pb-Sn-like eutectic diagram")
    ax.set_ylim(120, 360)
    ax.legend(loc="lower center")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "eutectic_diagram.png", dpi=120)

    # --- figure 2: cooling curves for 40 wt% Sn and 10 wt% Sn
    q = 0.5
    fig2, axs = plt.subplots(1, 2, figsize=(9.6, 4.4))
    for x, ax_ in zip((40.0, 10.0), axs):
        t, T, f = simulate_cooling(lambda T_: eq_solid_fraction(T_, x), constant_q(q), t_liquidus(x) + 40.0, 120.0)
        ax_.plot(t, T, "k-", label="equilibrium (lever rule)")
        ts_, Ts_, _ = simulate_cooling(lambda T_: scheil_solid_fraction(T_, x), constant_q(q), t_liquidus(x) + 40.0, 120.0)
        ax_.plot(ts_, Ts_, "r--", label="Scheil")
        ax_.set_xlabel("time (s)")
        ax_.set_ylabel("temperature (C)")
        ax_.set_title(f"{x:.0f} wt% Sn, heat loss {q} W/g")
        ax_.grid(alpha=0.3)
        ax_.legend(fontsize=8)
        print(f"{x:.0f} wt% Sn: liquidus {t_liquidus(x):.1f} C, eutectic arrest (equilibrium) {arrest_duration(t, T, T_E):.1f} s, "
              f"(Scheil) {arrest_duration(ts_, Ts_, T_E):.1f} s")
    fig2.tight_layout()
    fig2.savefig(OUT / "cooling_curves.png", dpi=120)

    # --- figure 3: coring (Scheil solid composition) and cooling-rate sensitivity
    x0 = 10.0
    fe_scheil = scheil_eutectic_fraction(x0)
    print(f"x0 = {x0:.0f}: Scheil eutectic fraction {100 * fe_scheil:.1f} %, equilibrium 0 %; core {K * x0:.2f}, rim up to {C_ALPHA:.0f} wt% Sn")
    fig3, axs3 = plt.subplots(1, 2, figsize=(9.6, 4.4))
    fs = np.linspace(0, 1.0 - fe_scheil, 200)
    axs3[0].plot(fs, scheil_solid_composition(1 - fs, x0), "r-", label="Scheil (fast)")
    axs3[0].axhline(x0, color="k", label="equilibrium (slow, after full diffusion)")
    axs3[0].set_xlabel("fraction solid when this layer formed (centre to edge)")
    axs3[0].set_ylabel("Sn in the solid layer (wt%)")
    axs3[0].set_title("Coring: the dendrite centre is Sn-poor")
    axs3[0].legend(fontsize=8)
    axs3[0].grid(alpha=0.3)
    qs = np.logspace(-2.5, 1.5, 17)
    fes = []
    for qq in qs:
        tf = freezing_time(x0, qq)
        fes.append(100 * clyne_kurz_eutectic_fraction(x0, alpha_prime(back_diffusion_alpha(tf))))
    axs3[1].semilogx(qs / C_HEAT, fes, "o-")
    axs3[1].axhline(100 * fe_scheil, color="r", ls="--", label="Scheil limit")
    axs3[1].set_xlabel("cooling rate of the liquid, q/c (K/s)")
    axs3[1].set_ylabel("eutectic in a 10 wt% Sn alloy (%)")
    axs3[1].set_title("Equilibrium predicts 0 %; fast cooling creates eutectic")
    axs3[1].legend(fontsize=8)
    axs3[1].grid(alpha=0.3)
    fig3.tight_layout()
    fig3.savefig(OUT / "scheil_and_rate.png", dpi=120)
    for qq, fe in zip(qs[::4], fes[::4]):
        print(f"  q = {qq:.4g} W/g (liquid rate {qq / C_HEAT:.3g} K/s): eutectic {fe:.2f} %")

## Where this stops
A lumped model has one temperature for the whole casting; real castings freeze from the wall inwards.
Constant k, straight lines and no undercooling are simplifications. Treat every number here as
a teaching illustration, never as design data for a real solder or casting.

## Coding tasks
1. **Easy.** Run the lesson and read the printed arrests for 40 and 10 wt% Sn; then change `L_HEAT` to 100 J/g and predict how the 40 wt% Sn arrest changes before you check it.
2. **Stretch.** Use `simulate_cooling` with `newtonian_q(0.002)` for 40 wt% Sn and plot the cooling curve; explain why the slope in the liquid is not constant.
3. **Explain.** Explain in your own words why the eutectic fraction for 10 wt% Sn is 0 at slow cooling, 5.8 percent at 5 W/g and approaches 7.2 percent at very fast cooling.

## Assessment
- **A. Conceptual.** A 10 wt% Sn alloy is single-phase alpha in the equilibrium diagram. Why does a fast-cooled joint of this alloy still contain some eutectic?
- **B. Numerical.** Using the Scheil equation, find the liquid composition for a 10 wt% Sn alloy when 90 percent has frozen, and then the fraction of eutectic it forms.
- **C. Graph and simulation interpretation.** On the eutectic-versus-cooling-rate graph, at which cooling rates does the eutectic stay zero, and what limit does the curve approach at high rates?
- **D. Design and optimisation challenge.** Design: choose the smallest whole-number Sn content whose liquid fraction is 60 percent at or below 230 C, and state the temperature. Why is the result not valid for fast-cooled solder?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("partition coefficient K = 19/62 = 0.306 and liquidus of 40 wt% Sn is 234.1 C",
                   abs(L.K - 19.0 / 62.0) < 1e-12 and abs(L.t_liquidus(40.0) - 234.09) < 0.01))
    # constant ratio of solid to liquid composition on every tie line
    ratios = [((L.T_PB - T) / L.M_S) / ((L.T_PB - T) / L.M_L) for T in (200.0, 250.0, 300.0)]
    checks.append(("both lines start at pure Pb, so C_solid / C_liquid = K on every tie line", np.allclose(ratios, L.K)))

    # hand calculation: pro-eutectic alpha and eutectic mixture for 40 wt% Sn
    pe = (62.0 - 40.0) / (62.0 - 19.0)
    checks.append((f"worked example: 40 wt% Sn just above 183 C is {100 * pe:.1f} percent pro-eutectic alpha and {100 * (1 - pe):.1f} percent liquid that becomes eutectic",
                   abs(L.pro_eutectic_fraction(40.0) - pe) < 1e-12 and abs(L.eq_solid_fraction(183.0 + 1e-7, 40.0) - pe) < 1e-6
                   and round(100 * pe, 1) == 51.2))
    checks.append(("eutectic composition 62 wt% Sn melts at one temperature (no band): f_s jumps 0 to 1 at 183 C",
                   L.eq_solid_fraction(183.1, 62.0) == 0.0 and L.eq_solid_fraction(183.0, 62.0) == 1.0))
    checks.append(("an alloy below 19 wt% Sn is fully solid after its solidus, with no eutectic",
                   L.eq_solid_fraction(L.t_solidus(10.0) - 0.01, 10.0) == 1.0 and L.t_solidus(10.0) > L.T_E))

    # --- the cooling simulation, constant heat loss q
    x, q = 40.0, 0.5
    tl = L.t_liquidus(x)
    T0, T_end = tl + 40.0, 120.0
    fs_eq = lambda T: L.eq_solid_fraction(T, x)
    t, T, f = L.simulate_cooling(fs_eq, L.constant_q(q), T0, T_end)
    t_liq = L.time_to_reach(t, T, tl - 1e-9)
    hand = L.C_HEAT * 40.0 / q
    checks.append((f"liquid cools at q/c: reaching the liquidus takes c*40/q = {hand:.1f} s (got {t_liq:.2f})", abs(t_liq - hand) < 0.05))
    arrest_hand = L.L_HEAT * (1.0 - pe) / q
    arrest = L.arrest_duration(t, T, L.T_E)
    checks.append((f"worked example: eutectic arrest lasts L*(1 - {pe:.3f})/q = {arrest_hand:.1f} s (simulated {arrest:.1f} s)",
                   abs(arrest - arrest_hand) < 0.2 and round(arrest_hand, 1) == 48.8))
    # energy conservation: heat removed = sensible + latent
    heat_removed = q * t[-1]
    heat_hand = L.C_HEAT * (T0 - T[-1]) + L.L_HEAT * 1.0
    checks.append((f"energy: heat removed q*t = {heat_removed:.2f} J/g equals c*dT + L = {heat_hand:.2f} J/g",
                   abs(heat_removed - heat_hand) < 2 * q * (t[1] - t[0])))
    checks.append(("total latent heat released is L (frozen fraction ends at 1)", abs(L.L_HEAT * f[-1] - L.L_HEAT) < 1e-6))
    # frozen fraction from the energy balance reproduces the lever rule
    pick = [i for i in range(len(T)) if 190.0 < T[i] < tl - 0.5][::37]
    err = max(abs(f[i] - L.eq_solid_fraction(T[i], x)) for i in pick)
    checks.append((f"simulated frozen fraction equals the lever rule between liquidus and 183 C (max error {err:.1e})", len(pick) > 5 and err < 1e-6))
    # slope change at the liquidus: rate = q / (c + L |d fs / dT|)
    Tm = 225.0
    dfs = (L.eq_solid_fraction(Tm + 0.01, x) - L.eq_solid_fraction(Tm - 0.01, x)) / 0.02
    rate_hand = q / (L.C_HEAT - L.L_HEAT * dfs)
    rate_sim = 2.0 / (L.time_to_reach(t, T, Tm - 1.0) - L.time_to_reach(t, T, Tm + 1.0))
    checks.append((f"slope change: rate at 225 C is {rate_hand:.2f} K/s (hand) and {rate_sim:.2f} K/s (simulated), slower than q/c = {q / L.C_HEAT:.1f}",
                   abs(rate_sim - rate_hand) / rate_hand < 0.02 and rate_hand < q / L.C_HEAT))

    # Newtonian cooling of the liquid against the exact exponential
    h = 0.002
    t2, T2, _ = L.simulate_cooling(fs_eq, L.newtonian_q(h), T0, 120.0)
    t_chk = 5.0
    T_exact = 25.0 + (T0 - 25.0) * math.exp(-h * t_chk / L.C_HEAT)
    T_num = float(np.interp(t_chk, t2, T2))
    checks.append((f"Newtonian cooling of the liquid matches the exponential (exact {T_exact:.2f} C, simulated {T_num:.2f} C)", abs(T_exact - T_num) < 0.1))

    # --- Scheil
    k = L.K
    fl = 0.1
    closed = 10.0 * fl ** (k - 1.0)
    rk4 = L.scheil_by_integration(10.0, 0.9)
    checks.append((f"Scheil: C_L = C0 f_L^(k-1) = {closed:.2f} wt% Sn at f_L = 0.1 equals the numerical mass-balance integration ({rk4:.2f})",
                   abs(closed - rk4) < 1e-4 and abs(closed - 49.38) < 0.01))
    mb = L.scheil_mass_balance(10.0, 0.9)
    checks.append((f"Scheil conserves Sn: solid plus liquid carry {mb:.6f} of the total", abs(mb - 1.0) < 1e-6))
    checks.append(("Scheil start: the first solid has composition k*C0 = 3.06 wt% Sn for C0 = 10",
                   abs(float(L.scheil_solid_composition(1.0, 10.0)) - 3.0645) < 1e-3 and abs(float(L.scheil_liquid_composition(1.0, 10.0)) - 10.0) < 1e-12))
    fe = L.scheil_eutectic_fraction(10.0)
    checks.append((f"coring and failure of equilibrium: Scheil makes {100 * fe:.1f} percent eutectic in a 10 wt% Sn alloy where the lever rule says 0",
                   abs(fe - 0.0720) < 5e-4 and L.eq_solid_fraction(L.T_E + 0.01, 10.0) == 1.0))
    checks.append(("segregation ratio: rim / core Sn = C_E / C0 = 6.2 for C0 = 10",
                   abs(19.0 / float(L.scheil_solid_composition(1.0, 10.0)) - 6.2) < 1e-9))
    # Scheil cooling curve has an extra arrest at 183 C that the equilibrium curve lacks
    ts, Ts, _ = L.simulate_cooling(lambda T_: L.scheil_solid_fraction(T_, 10.0), L.constant_q(q), L.t_liquidus(10.0) + 40.0, 120.0)
    te, Te, _ = L.simulate_cooling(lambda T_: L.eq_solid_fraction(T_, 10.0), L.constant_q(q), L.t_liquidus(10.0) + 40.0, 120.0)
    a_s, a_e = L.arrest_duration(ts, Ts, L.T_E), L.arrest_duration(te, Te, L.T_E)
    checks.append((f"Scheil cooling curve shows a 183 C arrest of {a_s:.1f} s (= L*{fe:.3f}/q); equilibrium shows {a_e:.1f} s",
                   abs(a_s - L.L_HEAT * fe / q) < 0.2 and a_e < 0.05))
    # Clyne-Kurz limits
    checks.append(("back-diffusion limits: a' = 0 gives Scheil, a' = 0.5 gives the lever rule (40 wt% Sn: 48.8 percent eutectic)",
                   abs(L.clyne_kurz_eutectic_fraction(10.0, 0.0) - fe) < 1e-9
                   and abs(L.clyne_kurz_eutectic_fraction(40.0, 0.5) - L.eutectic_mixture_fraction(40.0)) < 1e-9
                   and abs(L.alpha_prime(1e-6)) < 1e-5 and abs(L.alpha_prime(1e4) - 0.5) < 1e-3))
    # cooling-rate sensitivity (the failure case)
    res = {}
    for qq in (0.005, 0.5, 5.0, 50.0):
        tf = L.freezing_time(10.0, qq)
        res[qq] = 100 * L.clyne_kurz_eutectic_fraction(10.0, L.alpha_prime(L.back_diffusion_alpha(tf)))
    checks.append((f"slow cooling follows the diagram: eutectic {res[0.005]:.1f} % at 0.005 W/g and {res[0.5]:.1f} % at 0.5 W/g",
                   res[0.005] == 0.0 and res[0.5] == 0.0))
    checks.append((f"fast cooling breaks it: eutectic {res[5.0]:.1f} % at 5 W/g and {res[50.0]:.1f} % at 50 W/g (Scheil limit 7.2 %)",
                   abs(res[5.0] - 5.8) < 0.1 and abs(res[50.0] - 7.07) < 0.05 and res[50.0] <= 100 * fe))
    checks.append(("homogenising time scale lambda^2/D = 400 s", abs(L.LAMBDA ** 2 / L.D_SOLID - 400.0) < 1e-6))

    # --- mini-project design
    t30 = L.temperature_for_liquid_fraction(30.0, 0.6)
    checks.append((f"design: 30 wt% Sn is 60 percent liquid at {t30:.1f} C (lever rule check)",
                   abs(L.liquid_fraction_eq(30.0, t30) - 0.6) < 1e-9 and abs(t30 - 230.57) < 0.01))
    best = next(c for c in range(20, 61) if L.temperature_for_liquid_fraction(float(c), 0.6) <= 230.0)
    checks.append((f"design: smallest whole wt% Sn that is 60 percent liquid at or below 230 C is {best}", best == 31))
    checks.append(("edge: 50 wt% Sn is already 72 percent liquid just above 183 C, so 60 percent is only reached at the 183 C arrest",
                   abs(L.liquid_fraction_eq(50.0, 183.0 + 1e-7) - 0.7209) < 1e-3 and L.temperature_for_liquid_fraction(50.0, 0.6) == L.T_E))
    checks.append(("30 wt% Sn: eutectic fraction 25.6 percent in equilibrium and 35.1 percent with Scheil freezing",
                   round(100 * L.eutectic_mixture_fraction(30.0), 1) == 25.6 and round(100 * L.scheil_eutectic_fraction(30.0), 1) == 35.1))
    t31 = L.temperature_for_liquid_fraction(31.0, 0.6)
    checks.append((f"design: 31 wt% Sn is 60 percent liquid at {t31:.1f} C", round(t31, 1) == 227.4))
    ap = L.alpha_prime(L.back_diffusion_alpha(L.freezing_time(10.0, 10 ** 0.7)))
    checks.append(("app challenge: 10 wt% Sn at q = 10^0.7 W/g makes at least 5 percent eutectic",
                   L.clyne_kurz_eutectic_fraction(10.0, ap) >= 0.05))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
